In [ ]:
import os
import json
import pandas as pd
from mistralai.client import Mistral
import time
from tqdm import tqdm

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

BASE_PATH = "../"

In [3]:
def group_segments(segments, max_chars=750, hard_limit_chars=1500):
    chunks = []
    current_chunk = ""
    sentence_endings = ('.', '!', '?', '...')
    
    for seg in segments:
        text = seg['text'].strip()
        if not text: continue
        
        current_chunk += " " + text
        
        soft_split = len(current_chunk) >= max_chars and text.endswith(sentence_endings)
        hard_split = len(current_chunk) >= hard_limit_chars
        
        if soft_split or hard_split:
            chunks.append(current_chunk.strip())
            current_chunk = ""
            
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:
def clean_text_with_mistral(client, raw_text):
    """
    Отправляет сырой текст в Mistral для очистки.
    """
    prompt = f"""### РОЛЬ
Ты -- ведущий технический редактор образовательных материалов в области Computer Science и Machine Learning. 
Твоя специализация -- превращать грязные транскрипции лекций в локаничный профессиональный текст.

### ИНСТРУКЦИИ ПО ОБРАБОТКЕ
Выполни очистку текста, строго следуя алгоритму:

1. ОЧИСТКА РЕЧИ: Удали все слова-паразиты (эээ, ну, типа, как бы, значит, вот, ах, окей), заикания и повторы слов.
2. ИСПРАВЛЕНИЕ ОШИБОК: Исправь грамматические и пунктуационные ошибки, возникшие при распознавании речи (ASR). Восстанови логическую структуру предложений.
3. ПОВЫШЕНИЕ ИНФОРМАТИВНОСТИ: Удали мета-комментарии лектора, не несущие знаний (например: "слышно ли меня?", "перейдем к следующему слайду", "как-то странно звучит", организационные моменты). 
4. ПЕРЕВОД ТЕРМИНОВ: Если технический термин написан кириллицей, замени его на общепринятый английский эквивалент (например: "трэйн" -> "train", "инференс" -> "inference", "кернел" -> "kernel", "хост" -> "host", "девайс" -> "device", "мемкопи" -> "memcpy", "гряд" -> "grid", "юнифаед мемори" -> "Unified Memory").
5. СЖАТИЕ: Сократи текст, удалив "воду", но сохрани на 100% всю фактическую и техническую информацию.

### ОГРАНИЧЕНИЯ (Negative Constraints)
- НЕ добавляй вводных фраз ("Вот очищенный текст", "Результат:").
- НЕ добавляй примечаний от редактора или объяснений изменений.
- НЕ меняй смысл высказывания и не добавляй информации, которой не было в оригинале.
- НЕ заключай ответ в кавычки.
- Выдавай ТОЛЬКО финальный отредактированный текст.

### ПРИМЕР
Вход: "Ну, ааа, значит, мы тут выделяем, как его, кудомолок, на девайсе, эээ, чтобы потом сделать мемкопи с хоста."
Выход: "Выделяется память на device с помощью cudaMalloc для последующего выполнения memcpy с host."

### СЫРОЙ ТЕКСТ ДЛЯ ОБРАБОТКИ:
---
{raw_text}
---

### ОЧИЩЕННЫЙ ТЕКСТ:"""

    try:
        chat_response = client.chat.complete(
            model="mistral-small",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return chat_response.choices[0].message.content.strip()
    except Exception as e:
        print(f"API error: {e}")
        return None

In [ ]:
import os
import pandas as pd
import json
import random


RAW_DIR = BASE_PATH + "data/raw"
TRANSCRIPTIONS_BASE = BASE_PATH + "data/transcriptions"
CLEANING_DIR = BASE_PATH + "data/cleaning"
os.makedirs(CLEANING_DIR, exist_ok=True)
subdirs = sorted([d for d in os.listdir(TRANSCRIPTIONS_BASE) if os.path.isdir(os.path.join(TRANSCRIPTIONS_BASE, d))])
subdirs = ['00', '01', '02', '03', '04', '05', '06']
def generate_cleaning_transcription_dataset(json_input_path, csv_output_path):
    with open(json_input_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    text_chunks = group_segments(raw_data, max_chars=750)
    
    dataset_rows = []
    print(f"Processing {json_input_path}")
    
    for chunk in tqdm(text_chunks, desc="Cleaning chunks"):
        client = Mistral(api_key=MISTRAL_API_KEY, timeout_ms=4000)
        clean_chunk = None
        while clean_chunk is None:
            clean_chunk = clean_text_with_mistral(client, chunk)
        
        dataset_rows.append({
            "raw_text": chunk,
            "target_text": clean_chunk
        })
        time.sleep(random.uniform(4, 7))
        
    df = pd.DataFrame(dataset_rows)
    df.to_csv(csv_output_path, index=False)
    return len(df)

for subdir in tqdm(subdirs, desc="Generating Speech Cleaning Dataset"):
    json_path = os.path.join(TRANSCRIPTIONS_BASE, subdir, "lecture_raw.json")
    csv_path = os.path.join(CLEANING_DIR, f"{subdir}.csv")
    
    if os.path.exists(json_path):
        num_rows = generate_cleaning_transcription_dataset(json_path, csv_path)
    else:
        print(f"Error: file {json_path} not found")

Generating Speech Cleaning Dataset:   0%|          | 0/7 [00:00<?, ?it/s]

Processing ../data/transcriptions/00/lecture_raw.json


API error: The read operation timed out


Generating Speech Cleaning Dataset:  14%|█▍        | 1/7 [11:00<1:06:05, 660.99s/it]

Processing ../data/transcriptions/01/lecture_raw.json


...

Processing ../data/transcriptions/05/lecture_raw.json


API error: The read operation timed out


Generating Speech Cleaning Dataset:  86%|████████▌ | 6/7 [1:11:50<13:10, 790.04s/it]

Processing ../data/transcriptions/06/lecture_raw.json


Generating Speech Cleaning Dataset: 100%|██████████| 7/7 [1:22:47<00:00, 709.70s/it]
